# 05 · Evaluación con ventana creciente

**Qué hace:** para cada fuente (Stuttgart y ÁUREA) simula un reentrenamiento semanal. En cada punto de corte `d` entrena con todo lo anterior y pronostica los 8 días siguientes con el modelo (P50 y P90) y con los dos baselines. No calcula métricas: eso es el cuaderno 06.

**Decisiones que aplica:** D-10 (unos 52 cortes semanales, ventana creciente), D-11 (la configuración congelada en el 04, **sin tocarla**), D-06 (hiperparámetros elegidos una sola vez, también para ÁUREA), D-18 (se guarda la configuración y la huella de los datos).

| Fuente | Entrenamiento | Prueba (objetivos) | Papel |
|---|---|---|---|
| Stuttgart | desde 2013-10-04, creciente | 2014-11-08 a 2015-11-07 | Experimento principal (real) |
| ÁUREA | primer año, creciente | 2025-11-18 a 2026-11-17 | Control (sintético) |

**La prueba se ejecuta una sola vez.** Antes de empezar, el cuaderno comprueba que `configuracion/congelada.json` está en el repositorio y sin cambios desde su commit.

**Necesita:** `salidas/stuttgart/` (01) y `salidas/aurea/` (02), y `configuracion/congelada.json` (04). **Deja:** `salidas/<fuente>/predicciones.parquet` y `predicciones_meta.json`.

In [1]:
# Parámetros
import json, hashlib, subprocess, platform, time
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "cuadernos" else Path.cwd()
CONF = RAIZ / "configuracion"
N_CORTES = 52                    # cortes semanales por fuente (D-10)
H_MAX = 8
SUBDIR = ""
FUENTES = {
    # fuente: (primer corte, último día objetivo)
    "stuttgart": ("2014-11-07", "2015-11-07"),   # prueba de D-10: 2014-11-08 a 2015-11-07
    "aurea":     ("2025-11-17", "2026-11-17"),   # segundo año de la serie; el primero entrena
}

## 1. La configuración debe estar congelada y subida
Se comprueba con git: el archivo existe en el historial y no tiene cambios locales. Así, la configuración se fijó antes de ver la prueba.

In [2]:
ultimo_commit = subprocess.run(["git", "log", "-1", "--format=%H %cs", "--", str(CONF / "congelada.json")], capture_output=True, encoding="utf-8", cwd=RAIZ).stdout.strip()
sin_cambios = subprocess.run(["git", "diff", "--quiet", "HEAD", "--", str(CONF / "congelada.json")], cwd=RAIZ).returncode == 0
print("último commit de congelada.json:", ultimo_commit or "(ninguno)", "| sin cambios locales:", sin_cambios)
assert ultimo_commit and sin_cambios, "congelada.json debe estar commiteada y sin cambios antes de ejecutar la prueba"

congelada = json.loads((CONF / "congelada.json").read_text(encoding="utf-8"))
congelada_sha = hashlib.sha256((CONF / "congelada.json").read_bytes()).hexdigest()
congelada["p50"], congelada["hiperparametros"], congelada_sha[:12]

último commit de congelada.json: e656657ec4b05ed959e2cbaac5ec40a0c6081aec 2026-11-20 | sin cambios locales: True


({'loss': 'quantile', 'quantile': 0.5},
 {'learning_rate': 0.05,
  'max_iter': 200,
  'max_leaf_nodes': 15,
  'min_samples_leaf': 50},
 '92fd1ad05e37')

## 2. Función de evaluación
`evaluar(fuente)` hace, para cada corte semanal `d`:
1. **Entrenamiento:** las filas con objetivo ≤ `d` (ventana creciente). Equivale a `filas_entrenamiento(historial, calendario, d)`; se calcula la tabla una vez y se recorta, y se comprueba la equivalencia en un corte.
2. **Prueba:** las filas de ese corte, con sus 8 objetivos, dentro de la ventana de la fuente. Los días cerrados no son objetivo.
3. **Modelo:** P50 con la pérdida y los hiperparámetros congelados, P90 con `quantile=0.9` y los mismos hiperparámetros. Garantías de CONTRATO 4.2: recorte a 0 y `p90 ≥ p50`.
4. **Baselines:** ingenuo estacional (P50), media del mismo día y P90 del baseline (percentil 90 de los últimos 8 mismos días; con menos de 4 valores, igual al P50).

In [3]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from rc_pronostico.variables import (LISTA_VARIABLES, CATEGORICAS, VERSION_VARIABLES, codigos_producto, filas_entrenamiento)
from rc_pronostico.baselines import baselines

assert congelada["variables"] == LISTA_VARIABLES and congelada["version_variables"] == VERSION_VARIABLES

def regresor(loss, quantile=None):
    kw = {"loss": loss, **({"quantile": quantile} if quantile is not None else {})}
    return HistGradientBoostingRegressor(**kw, **congelada["hiperparametros"], categorical_features=CATEGORICAS,
                                         early_stopping=congelada["early_stopping"], random_state=congelada["random_state"])

def cargar(fuente):
    d = RAIZ / "salidas" / fuente
    return (pd.read_parquet(d / "historial.parquet"), json.loads((d / "calendario.json").read_text(encoding="utf-8")),
            hashlib.sha256((d / "historial.parquet").read_bytes()).hexdigest())

def evaluar(fuente):
    primer_corte, fin = (pd.Timestamp(x) for x in FUENTES[fuente])
    historial, calendario, huella = cargar(fuente)
    cal = {"dias_cerrados": calendario["dias_cerrados"], "feriados": calendario["feriados"]}
    codigos = codigos_producto(historial)
    todas = filas_entrenamiento(historial, cal, fin, codigos)          # objetivos hasta el fin de la prueba
    cortes = [primer_corte + pd.Timedelta(days=7 * k) for k in range(N_CORTES)]
    salida, t0 = [], time.time()
    for k, d in enumerate(cortes):
        ent = todas[todas.fecha_objetivo <= d]
        prueba = todas[(todas.fecha_corte == d) & (todas.fecha_objetivo > d) & (todas.fecha_objetivo <= fin)].copy()
        assert ent.fecha_objetivo.max() <= d and (prueba.fecha_objetivo > d).all()   # sin fuga, en cada corte
        if k == 0:   # el recorte equivale a recalcular con el historial cortado en d
            directo = filas_entrenamiento(historial, cal, d, codigos)
            pd.testing.assert_frame_equal(ent.reset_index(drop=True), directo, check_dtype=False)
        m50 = regresor(**congelada["p50"]).fit(ent[LISTA_VARIABLES], ent.y)
        m90 = regresor(**congelada["p90"]).fit(ent[LISTA_VARIABLES], ent.y)
        p50 = np.clip(m50.predict(prueba[LISTA_VARIABLES]), 0, None)
        p90 = np.maximum(np.clip(m90.predict(prueba[LISTA_VARIABLES]), 0, None), p50)
        bl = baselines(historial, cal, prueba)
        p90_bl = np.where(bl.n_mismo_dia >= 4, bl.p90_mismo_dia, bl.ingenuo)
        salida.append(prueba[["producto_id", "fecha_corte", "fecha_objetivo", "h", "dia_semana", "y"]].assign(
            p50=p50, p90=p90, ingenuo=bl.ingenuo.values, media4=bl.media_mismo_dia.values,
            p90_baseline=np.maximum(p90_bl, bl.ingenuo.values), n_entrenamiento=len(ent)))
    pred = pd.concat(salida, ignore_index=True)
    pred.insert(0, "fuente", fuente)
    meta = {"fuente": fuente, "n_cortes": len(cortes), "primer_corte": str(cortes[0].date()), "ultimo_corte": str(cortes[-1].date()),
            "fin_prueba": str(fin.date()), "filas": len(pred), "huella_historial_sha256": huella,
            "congelada_sha256": congelada_sha, "ultimo_commit_congelada": ultimo_commit,
            "version_variables": VERSION_VARIABLES, "semilla": congelada["random_state"],
            "python": platform.python_version(),
            "segundos": round(time.time() - t0)}
    return pred, meta

## 3. Stuttgart (experimento principal)

In [4]:
pred_s, meta_s = evaluar("stuttgart")
print(json.dumps({k: meta_s[k] for k in ("n_cortes", "primer_corte", "ultimo_corte", "fin_prueba", "filas", "segundos")}))
pred_s.head()

{"n_cortes": 52, "primer_corte": "2014-11-07", "ultimo_corte": "2015-10-30", "fin_prueba": "2015-11-07", "filas": 2884, "segundos": 54}


,fuente,producto_id,fecha_corte,fecha_objetivo,h,dia_semana,y,p50,p90,ingenuo,media4,p90_baseline,n_entrenamiento
0,stuttgart,CALAMARI,2014-11-07,2014-11-08,1,5,9.0,5.851975,8.439835,5.0,5.25,8.0,20384
1,stuttgart,FISCH,2014-11-07,2014-11-08,1,5,9.0,5.896234,8.193717,3.0,3.75,6.0,20384
2,stuttgart,GARNELEN,2014-11-07,2014-11-08,1,5,20.0,12.179450,19.804624,11.0,13.00,15.6,20384
3,stuttgart,HAEHNCHEN,2014-11-07,2014-11-08,1,5,51.0,48.144217,51.674467,52.0,42.75,52.0,20384
4,stuttgart,KOEFTE,2014-11-07,2014-11-08,1,5,65.0,31.064760,47.333356,34.0,38.75,42.8,20384


## 4. ÁUREA (control, sintético)
Mismo código y misma configuración: los hiperparámetros se eligieron una sola vez, en Stuttgart (D-06).

In [5]:
pred_a, meta_a = evaluar("aurea")
print(json.dumps({k: meta_a[k] for k in ("n_cortes", "primer_corte", "ultimo_corte", "fin_prueba", "filas", "segundos")}))
pred_a.head()

{"n_cortes": 52, "primer_corte": "2025-11-17", "ultimo_corte": "2026-11-09", "fin_prueba": "2026-11-17", "filas": 8280, "segundos": 102}


,fuente,producto_id,fecha_corte,fecha_objetivo,h,dia_semana,y,p50,p90,ingenuo,media4,p90_baseline,n_entrenamiento
0,aurea,00000000-0000-4000-8000-000000000431,2025-11-17,2025-11-18,1,1,20.0,19.099730,26.520272,18.0,18.50,23.8,52480
1,aurea,00000000-0000-4000-8000-000000000432,2025-11-17,2025-11-18,1,1,30.0,27.151268,37.679984,34.0,31.00,34.0,52480
2,aurea,00000000-0000-4000-8000-000000000433,2025-11-17,2025-11-18,1,1,28.0,26.962442,35.229167,27.0,26.75,30.3,52480
3,aurea,00000000-0000-4000-8000-000000000434,2025-11-17,2025-11-18,1,1,16.0,17.134986,24.920844,20.0,18.75,20.0,52480
4,aurea,00000000-0000-4000-8000-000000000435,2025-11-17,2025-11-18,1,1,9.0,7.232446,11.545588,5.0,7.00,8.9,52480


## 5. Salidas

In [6]:
for fuente, pred, meta in (("stuttgart", pred_s, meta_s), ("aurea", pred_a, meta_a)):
    d = RAIZ / "salidas" / SUBDIR / fuente if SUBDIR else RAIZ / "salidas" / fuente
    d.mkdir(parents=True, exist_ok=True)
    pred.to_parquet(d / "predicciones.parquet", index=False)
    (d / "predicciones_meta.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")
resumen = pd.DataFrame({"filas": [len(pred_s), len(pred_a)], "platos": [pred_s.producto_id.nunique(), pred_a.producto_id.nunique()],
                        "primer objetivo": [pred_s.fecha_objetivo.min().date(), pred_a.fecha_objetivo.min().date()],
                        "último objetivo": [pred_s.fecha_objetivo.max().date(), pred_a.fecha_objetivo.max().date()]}, index=["stuttgart", "aurea"])
resumen

,filas,platos,primer objetivo,último objetivo
stuttgart,2884,7,2014-11-08,2015-11-07
aurea,8280,20,2025-11-18,2026-11-17


## Verificaciones

In [7]:
for fuente, pred, meta in (("stuttgart", pred_s, meta_s), ("aurea", pred_a, meta_a)):
    fin = pd.Timestamp(FUENTES[fuente][1])
    assert pred.fecha_objetivo.max() <= fin and (pred.fecha_objetivo > pred.fecha_corte).all(), fuente
    assert pred.h.between(1, H_MAX).all() and pred.fecha_corte.nunique() == N_CORTES, fuente
    assert pred[["p50", "p90", "ingenuo", "media4", "p90_baseline", "y"]].notna().all().all(), fuente + ": hay predicciones faltantes"
    assert (pred.p50 >= 0).all() and (pred.p90 >= pred.p50).all() and (pred.p90_baseline >= pred.ingenuo).all(), fuente
    assert (pred.groupby("fecha_corte").n_entrenamiento.first().diff().dropna() > 0).all(), fuente + ": la ventana no crece"
    assert not pred.duplicated(["producto_id", "fecha_corte", "h"]).any(), fuente
    cerr = set(json.loads((RAIZ / "salidas" / fuente / "calendario.json").read_text(encoding="utf-8"))["dias_cerrados"])
    assert not set(pred.fecha_objetivo.dt.strftime("%Y-%m-%d")) & cerr, fuente + ": hay objetivos en días cerrados"
assert pred_s.producto_id.nunique() == 7 and pred_a.producto_id.nunique() == 20
print("OK · predicciones generadas con la configuración congelada", congelada_sha[:12])

OK · predicciones generadas con la configuración congelada 92fd1ad05e37
